In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# ClinicalDistill — Gemma-3-1B **QLoRA**

## What is QLoRA and why are we using it?

**LoRA**: loads the full model in float16 (~2GB), injects small trainable adapters.

**QLoRA** = Quantized LoRA. Same LoRA adapters, BUT the base model is loaded in **4-bit** instead of 16-bit.

### Why does this matter?
- 16-bit = 2 bytes per parameter → 1B params × 2 bytes = **~2GB VRAM**
- 4-bit  = 0.5 bytes per parameter → 1B params × 0.5 bytes = **~0.5GB VRAM**

For larger models (7B, 13B) this is the difference between fitting on a T4 or not.
For our 1B model it means faster loading and less memory pressure.

### Research value
We run QLoRA on the same dataset with the same LoRA config → compare:
- Training time (QLoRA is often faster)
- Memory usage  
- F1 score (does quantization hurt accuracy?)

This is **Experiment 3** in my research log and one row in the paper's results table.

In [2]:
import os

PROJECT_PATH = '/content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill'

print(os.listdir(PROJECT_PATH))
!wc -l {PROJECT_PATH}/data/train_fixed.jsonl {PROJECT_PATH}/data/test_fixed.jsonl

['README.md', 'requirements.txt', '.env', '.gitignore', 'data', 'schema', '.git', 'models']
  145 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/train_fixed.jsonl
   35 /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/data/test_fixed.jsonl
  180 total


In [3]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.6 MB/s eta 0:00:00


In [4]:
import torch

#QLoRA requires CUDA
print(torch.cuda.is_available())      # should be True
print(torch.cuda.get_device_name(0))  # should be Tesla T4
print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

True
Tesla T4
VRAM available: 15.6 GB


In [5]:
from datasets import load_dataset
import json

# train_fixed.jsonl: 145 examples (120 formal + 25 casual)
# test_fixed.jsonl:   35 examples (30 formal + 5 casual)
dataset = load_dataset("json", data_files={
    "train": f"{PROJECT_PATH}/data/train_fixed.jsonl",
    "test":  f"{PROJECT_PATH}/data/test_fixed.jsonl"
})

print(dataset)
print("\nSample example:")
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 145
    })
    test: Dataset({
        features: ['input', 'output', 'metadata'],
        num_rows: 35
    })
})

Sample example:
{'input': 'Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.', 'output': {'symptoms': ['chest pain', 'shortness of breath'], 'duration': ['2 hours', 'unspecified'], 'severity': ['severe', 'unspecified'], 'urgent': True}, 'metadata': {'clinical_domain': 'cardiac', 'generated_by': 'gpt-4o'}}


In [6]:
def format_example(example):
    output = example['output']
    if isinstance(output, str):
        output = json.loads(output)

    return {
        "text": f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{example['input']}</input>
<output>{json.dumps(output)}</output>"""
    }

dataset = dataset.map(format_example)
print("Formatted example:")
print(dataset["train"][0]["text"])

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

Map:   0%|          | 0/35 [00:00<?, ? examples/s]

Formatted example:
<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>Patient presents with chest pain radiating to the left arm and shortness of breath for the past 2 hours.</input>
<output>{"symptoms": ["chest pain", "shortness of breath"], "duration": ["2 hours", "unspecified"], "severity": ["severe", "unspecified"], "urgent": true}</output>


In [8]:
from huggingface_hub import login
import json
login()

## Load Model with 4-bit Quantization

This is the **only** difference from the LoRA notebook.

```python
BitsAndBytesConfig(
    load_in_4bit=True,          # load weights as 4-bit integers instead of 16-bit floats
    bnb_4bit_quant_type="nf4",  # NormalFloat4 — best accuracy for 4-bit (from QLoRA paper)
    bnb_4bit_compute_dtype=torch.float16,  # upcast to float16 for actual math operations
    bnb_4bit_use_double_quant=True,        # quantize the quantization constants too — saves ~0.4GB extra
)
```

### How it works
- Weights are STORED as 4-bit (saves memory)
- During forward pass, weights are temporarily UPCAST to float16 for computation
- Best of both worlds: small storage, accurate math

### Why nf4 specifically?
NF4 (NormalFloat4) was introduced in the QLoRA paper. It's designed for normally-distributed weights
(which neural network weights typically are). Better than regular int4 quantization.

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "google/gemma-3-1b-it"

#4 bit quantization config

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # store weights in 4-bit
    bnb_4bit_quant_type="nf4",             # NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16,  # compute in float16
    bnb_4bit_use_double_quant=True,        # double quantization for extra memory savings
)


tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print(f"Model loaded: {model_id}")
print(f"Parameters: {model.num_parameters():,}")

# Show memory saved vs float16
vram_used = torch.cuda.memory_allocated() / 1e9
print(f"VRAM used after loading: {vram_used:.2f} GB (vs ~2GB for float16 LoRA)")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model loaded: google/gemma-3-1b-it
Parameters: 999,885,952
VRAM used after loading: 1.35 GB (vs ~2GB for float16 LoRA)


## Prepare Model for QLoRA Training

`prepare_model_for_kbit_training` is a required step for QLoRA.
It does two things:
1. Converts all LayerNorm layers to float32 (for stable training)
2. Enables gradient checkpointing (trades compute for memory — recomputes activations instead of storing them)

Without this step, training a quantized model can produce NaN losses.

In [12]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# stablizes training on quantized model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,490,944 || all params: 1,001,376,896 || trainable%: 0.1489


In [14]:
# Training

from trl import SFTTrainer, SFTConfig
import time

OUTPUT_DIR = f"{PROJECT_PATH}/models/gemma-qlora"

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,              # reduced from 7 — less overfitting risk
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 2x4 = 8
    learning_rate=2e-4,
    fp16=False,   # Gemma-3 uses BFloat16 internally — fp16 conflicts with 4-bit quant
    bf16=True,    # use bf16 instead — matches Gemma-3's native dtype
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

# Track training time for paper comparison
start_time = time.time()
print("Starting QLoRA training...")
trainer.train()
training_time = time.time() - start_time

print(f"Training complete!")
print(f"Training time: {training_time:.0f}s ({training_time/60:.1f} min)")
print("Note this number — compare against LoRA training time in research log")

Starting QLoRA training...


Step,Training Loss
10,2.181717
20,1.265304
30,0.713188
40,0.431955
50,0.337594
60,0.284390
70,0.290016
80,0.297027
90,0.268603


Training complete!
Training time: 248s (4.1 min)
Note this number — compare against LoRA training time in research log


In [16]:
SAVE_PATH = f"{PROJECT_PATH}/models/gemma-qlora-final"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"QLoRA model saved to: {SAVE_PATH}")

QLoRA model saved to: /content/drive/MyDrive/Projects/Clinical-Distill/ClinicalDistill/models/gemma-qlora-final


We reload with the same quantization config so memory usage stays low at inference too.


In [17]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

SAVE_PATH = f"{PROJECT_PATH}/models/gemma-qlora-final"

# Reload with same 4-bit config for memory-efficient inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    quantization_config=bnb_config,
    device_map="auto"
)
model_loaded = PeftModel.from_pretrained(base_model, SAVE_PATH)
tokenizer_loaded = AutoTokenizer.from_pretrained(SAVE_PATH)

print("QLoRA model loaded for inference!")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

QLoRA model loaded for inference!
VRAM used: 1.98 GB


In [18]:
def extract_clinical(text):
    # IDENTICAL prompt to LoRA notebook — same inference format
    prompt = f"""<instruction>
Extract symptoms from the clinical note below. Reply with ONLY valid JSON. No comments, no markdown, no extra text.
Format: {{"symptoms": ["s1", "s2"], "duration": ["d1", "d2"], "severity": ["sev1", "sev2"], "urgent": true/false}}
Use "unspecified" if duration or severity unknown. urgent=true only for chest pain, breathing difficulty, stroke, severe bleeding.
</instruction>
<input>{text}</input>
<output>"""

    inputs = tokenizer_loaded(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model_loaded.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer_loaded.eos_token_id
        )

    response = tokenizer_loaded.decode(outputs[0], skip_special_tokens=True)
    return response.split("<output>")[-1].strip()


# Quick sanity check — same 3 test cases as LoRA notebook
test_cases = [
    "Elderly patient complaining of severe headache for 2 days, blurred vision, and confusion.",
    "Child presents with mild fever since yesterday, runny nose and occasional cough.",
    "Patient reports sharp lower back pain for a week, worse when sitting, no fever."
]

for case in test_cases:
    print(f"INPUT: {case}")
    print(f"OUTPUT: {extract_clinical(case)}")
    print("-" * 60)

INPUT: Elderly patient complaining of severe headache for 2 days, blurred vision, and confusion.
OUTPUT: {"symptoms": ["headache", "blurred vision", "confusion"], "duration": ["2 days", "unspecified"], "severity": ["severe", "unspecified"], "urgent": true}</output>
------------------------------------------------------------
INPUT: Child presents with mild fever since yesterday, runny nose and occasional cough.
OUTPUT: {"symptoms": ["mild fever", "runny nose", "occasional cough"], "duration": ["since yesterday", "unspecified", "unspecified"], "severity": ["mild", "unspecified", "unspecified"], "urgent": false}</output>
------------------------------------------------------------
INPUT: Patient reports sharp lower back pain for a week, worse when sitting, no fever.
OUTPUT: {"symptoms": ["lower back pain", "sitting"], "duration": ["one week", "unspecified"], "severity": ["unspecified", "unspecified"], "urgent": false}</output>
------------------------------------------------------------


## Evaluation

Same eval code as Experiment 2. Same test set (35 examples).
Results here go directly into the paper's comparison table.

In [19]:
import json

def parse_output(text):
    try:
        text = text.strip()
        # Strip closing XML tag the model generates
        text = text.replace("</output>", "").strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip()), True
    except:
        return {}, False

def flatten_symptoms(symptoms):
    flat = []
    for s in symptoms:
        if isinstance(s, str):
            flat.append(s.lower())
        elif isinstance(s, dict):
            name = s.get("symptom") or s.get("name") or s.get("description") or ""
            if name:
                flat.append(name.lower())
    return flat

def symptom_f1(pred_symptoms, true_symptoms):
    pred_set = set(flatten_symptoms(pred_symptoms))
    true_set = set(flatten_symptoms(true_symptoms))
    tp = len(pred_set & true_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if precision + recall > 0 else 0
    return round(f1, 3)

def urgent_accuracy(pred, true):
    return pred.get("urgent") == true.get("urgent")

# Run eval on full test set
valid_json_count = 0
f1_scores = []
urgent_correct = 0
results = []

for example in dataset["test"]:
    raw_output = extract_clinical(example["input"])
    pred, is_valid = parse_output(raw_output)
    true = example["output"]

    f1 = 0
    if is_valid:
        valid_json_count += 1
        f1 = symptom_f1(pred.get("symptoms", []), true.get("symptoms", []))
        f1_scores.append(f1)
        if urgent_accuracy(pred, true):
            urgent_correct += 1

    results.append({
        "input": example["input"],
        "expected": true,
        "predicted": pred,
        "valid_json": is_valid,
        "f1": f1
    })

total = len(dataset["test"])
avg_f1 = sum(f1_scores) / len(f1_scores) if f1_scores else 0

print("=" * 50)
print("EVALUATION RESULTS — Gemma-3-1B QLoRA")
print("=" * 50)
print(f"Total test examples : {total}")
print(f"Valid JSON rate     : {valid_json_count}/{total} ({100*valid_json_count/total:.1f}%)")
print(f"Avg Symptom F1      : {avg_f1:.3f}")
print(f"Urgent Accuracy     : {urgent_correct}/{valid_json_count} ({100*urgent_correct/max(valid_json_count,1):.1f}%)")
print("=" * 50)

EVALUATION RESULTS — Gemma-3-1B QLoRA
Total test examples : 35
Valid JSON rate     : 35/35 (100.0%)
Avg Symptom F1      : 0.740
Urgent Accuracy     : 29/35 (82.9%)
